In [28]:
import time
import tiktoken
import torch
import torch.nn as nn

In [29]:
import torch
import torch.nn as nn

class MultiQueryAttentionKV(nn.Module):
  def __init__(self, d_in, d_out, context_length, num_heads, dropout=0.0, qkv_bias=False):
    super().__init__()
    assert d_out % num_heads == 0, "d_model must be divisible by num_heads"

    self.d_out = d_out
    self.num_heads = num_heads
    self.head_dim = d_out // num_heads

    self.W_query = nn.Linear(d_in, d_out, bias=False)
    self.W_key = nn.Linear(d_in, self.head_dim , bias=False)    # Single projection for k
    self.W_value = nn.Linear(d_in, self.head_dim , bias=False)  # Single projection for V

    self.out_proj = nn.Linear(d_out, d_out)

    self.dropout = nn.Dropout(dropout)

    self.register_buffer("cache_k", None, persistent=False)
    self.register_buffer("cache_v", None, persistent=False)
    self.ptr_current_pos = 0
    

  def forward(self, x, use_cache=False):
    batch_size, num_tokens, d_in =  x.shape

    # Query
    queries = self.W_query(x)    # (batch_size, num_tokens, d_out)
    
    # Unroll last dim : (batch_size, num_tokens, d_out)  ---> (batch_size, num_tokens, num_heads, head_dim)
    queries = queries.view(batch_size, num_tokens, self.num_heads, self.head_dim)
    # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
    queries = queries.transpose(1, 2)
    
    # Key & value
    keys = self.W_key(x)      # (batch_size, num_tokens, head_dim)
    values = self.W_value(x)  # (batch_size, num_tokens, head_dim)

    # Unroll last dim : (batch_size, num_tokens, head_dim)  ---> (batch_size, num_tokens, 1, head_dim)
    keys = keys.view(batch_size, num_tokens, 1, self.head_dim)     # only 1 head
    values = values.view(batch_size, num_tokens, 1, self.head_dim)  # only 1 head

    # Transpose: (batch_size, num_tokens, 1, head_dim) -> (batch_size, 1, num_tokens, head_dim)
    keys_new = keys.transpose(1, 2)
    values_new = values.transpose(1, 2)
    
    # K-V cache
    if use_cache:
      if self.cache_k is None:
        self.cache_k, self.cache_v = keys_new, values_new
      else:
        self.cache_k = torch.cat([self.cache_k, keys_new], dim=2)   # dim = 2, concatenate across num tokens
        self.cache_v = torch.cat([self.cache_v, values_new], dim=2)
      keys, values = self.cache_k, self.cache_v
    else:
      keys, values = keys_new, values_new


    # Now Repeat K and V to match the query head
    keys = keys.repeat(1, self.num_heads, 1, 1)  # (batch_size, num_heads, num_tokens, head_dim)
    values = values.repeat(1, self.num_heads, 1, 1)  # (batch_size, num_heads, num_tokens, head_dim)

    # Attn scores
    attn_scores = queries @ keys.transpose(2, 3)

    # Apply causal mask
    num_tokens_Q = queries.shape[-2]
    num_tokens_K = keys.shape[-2]
    device = queries.device
    if use_cache:
      q_positions = torch.arange(self.ptr_current_pos, self.ptr_current_pos + num_tokens_Q, device=device, dtype=torch.long)
      self.ptr_current_pos += num_tokens_Q
    else:
      q_positions = torch.arange(num_tokens_Q, device=device, dtype=torch.long)
      self.ptr_current_pos = 0
    k_positions = torch.arange(num_tokens_K, device=device, dtype=torch.long)
    mask = q_positions.unsqueeze(-1) < k_positions.unsqueeze(0)
    
    # Use the mask to fill attention scores
    attn_scores = attn_scores.masked_fill(mask, -torch.inf)

    # Attn weights
    attn_weights = torch.softmax(attn_scores / (keys.shape[-1]**0.5), dim=-1)

    # dropout
    attn_weights = self.dropout(attn_weights)

    # context vectors
    context_vector = attn_weights @ values  # (batch_size, num_heads, num_tokens, head_dim)

    # (batch_size, num_tokens, num_heads, head_dim)
    context_vector = context_vector.transpose(1, 2)

    # Combine heads, where self.d_out = self.num_heads * self.head_dim
    # (batch_size, num_tokens, num_heads, head_dim) ---> # (batch_size, num_tokens, d_out)
    context_vector = context_vector.contiguous().view(batch_size, num_tokens, self.d_out)

    context_vector = self.out_proj(context_vector)

    return context_vector
    
  def reset_cache(self):
    self.cache_k, self.cache_v = None, None
    self.ptr_current_pos = 0


In [30]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift


class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)


In [31]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiQueryAttentionKV(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x, use_cache=False):
        # Shortcut connection for attention block
        shortcut = x
        x = self.norm1(x)

        # x = self.att(x)   # Shape [batch_size, num_tokens, emb_size]
        ####################################################
        # NEW
        x = self.att(x, use_cache=use_cache)
        ####################################################

        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        # Shortcut connection for feed-forward block
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        return x


In [32]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        # self.trf_blocks = nn.Sequential(
        #    *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        ####################################################
        # NEW
        self.trf_blocks = nn.ModuleList(
            [TransformerBlock(cfg) for _ in range(cfg["n_layers"])])

        self.current_pos = 0
        ####################################################

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx, use_cache=False):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)

        # pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))

        ####################################################
        # NEW

        if use_cache:
            pos_ids = torch.arange(self.current_pos, self.current_pos + seq_len, device=in_idx.device, dtype=torch.long)
            self.current_pos += seq_len
        else:
            pos_ids = torch.arange(0, seq_len, device=in_idx.device, dtype=torch.long)
        pos_embeds = self.pos_emb(pos_ids).unsqueeze(0)
        ####################################################

        x = tok_embeds + pos_embeds  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_emb(x)

        # x = self.trf_blocks(x)
        ####################################################
        # NEW
        for blk in self.trf_blocks:
            x = blk(x, use_cache=use_cache)
        ####################################################

        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

    ####################################################
    # NEW
    def reset_kv_cache(self):
        for blk in self.trf_blocks:
            blk.att.reset_cache()
        self.current_pos = 0
    ####################################################



In [33]:
GPT_CONFIG_124M = {
        "vocab_size": 50257,     # Vocabulary size
        "context_length": 1024,  # Context length
        "emb_dim": 768,          # Embedding dimension
        "n_heads": 1,           # Number of attention heads
        "n_layers": 1,          # Number of layers
        "drop_rate": 0.1,        # Dropout rate
        "qkv_bias": False        # Query-Key-Value bias
    }

model = GPTModel(GPT_CONFIG_124M)

In [34]:
def generate_text_simple_cached(model, idx, max_new_tokens, context_size=None, use_cache=True):
  model.eval()

  ctx_len = context_size or model.pos_emb.num_embeddings

  with torch.no_grad():
    if use_cache:
      # Init cache with full prompt
      model.reset_kv_cache()  # reset the previous cache

      logits = model(idx[:, -ctx_len:], use_cache=True)
      
      print(logits.shape)
      
      print()
      print("inputs prompt is Cached is done")
      print()

      for i in range(max_new_tokens):

        # a) pick the token with the highest log-proability (greedy sampling)
            # (batch, n_token, vocab_size) becomes (batch, vocab_size)
        logits = logits[:, -1, :]
        # get the idx 
        next_idx = torch.argmax(logits, dim=1, keepdim=True)   # (batch, 1)

        # b) Append sampled index to the running sequence
        idx = torch.cat((idx, next_idx), dim=1)  # (batch, n_tokens+1)

        # c) feed the model only new token and not the whole concatenated idx
        logits = model(next_idx, use_cache=True)

        print(i+1, logits.shape)
    else:
      for _ in range(max_new_tokens):
        logits = model(idx[:, -ctx_len:], use_cache=False)

        # (batch, n_token, vocab_size) becomes (batch, vocab_size)
        logits = logits[:, -1, :]

        # get the idx 
        next_idx = torch.argmax(logits, dim=1, keepdim=True)   # (batch, 1)

        # Append sampled index to the running sequence
        idx = torch.cat((idx, next_idx), dim=1)  # (batch, n_tokens+1)
    return idx


In [35]:
tokenizer = tiktoken.get_encoding('gpt2')


start_context = "Hello, I am"
encoded = tokenizer.encode(start_context)
print("encoded:", encoded)

encoded_tensor = torch.tensor(encoded).unsqueeze(0)
print("encoded_tensor.shape:", encoded_tensor)

encoded: [15496, 11, 314, 716]
encoded_tensor.shape: tensor([[15496,    11,   314,   716]])


In [36]:
model.eval() 
model = GPTModel(GPT_CONFIG_124M)
max_new_tokens = 6  # we have max new token to generete is 6

out = generate_text_simple_cached(model=model, idx=encoded_tensor, max_new_tokens=6, context_size=GPT_CONFIG_124M['context_length'], use_cache=True)

print("Output:", out)
print("Output length:", len(out[0]))

torch.Size([1, 4, 50257])

inputs prompt is Cached is done

1 torch.Size([1, 1, 50257])
2 torch.Size([1, 1, 50257])
3 torch.Size([1, 1, 50257])
4 torch.Size([1, 1, 50257])
5 torch.Size([1, 1, 50257])
6 torch.Size([1, 1, 50257])
Output: tensor([[15496,    11,   314,   716, 27586, 42098, 41255,  8553, 27795, 46713]])
Output length: 10
